In [1]:
import numpy as np
import tifffile as tiff
import nibabel as nib

# -------- paths --------
ome_tiff_path = "EMMA2.ome.tif"
nii_path = "EMMA2.nii"
ome_tiff_path = "crop.ome.tif"
nii_path = "crop.nii"

# -------- load OME-TIFF --------
img = tiff.imread(ome_tiff_path)

print("Original shape:", img.shape)

# -------- handle channel axis --------
# Common cases:
# (C, Y, X)  -> move C to Z
# (Y, X, C)  -> move C to Z

if img.ndim != 3:
    raise ValueError("Expected a 3D OME-TIFF (channels + 2D image)")

# If channels first
if img.shape[0] <= 64:  # heuristic: channels usually smaller
    # (C, Y, X) -> (X, Y, Z)
    volume = np.transpose(img, (2, 1, 0))
# If channels last
else:
    # (Y, X, C) -> (X, Y, Z)
    volume = np.transpose(img, (1, 0, 2))

print("NIfTI shape (X, Y, Z):", volume.shape)

volume = volume[:, :, 9:-1]

print("After slicing (X, Y, Z):", volume.shape)

# -------- create NIfTI --------
affine = np.eye(4)  # identity affine (edit if you have spacing info)
nii = nib.Nifti1Image(volume.astype(np.float32), affine)

# -------- save --------
nib.save(nii, nii_path)

print(f"Saved NIfTI to {nii_path}")


Original shape: (48, 391, 390)
NIfTI shape (X, Y, Z): (390, 391, 48)
After slicing (X, Y, Z): (390, 391, 38)
Saved NIfTI to crop.nii
